# EDA — Clusters FaaS (Azure Trace 2019)

Analyse exploratoire des 4 clusters HDBSCAN retenus pour la baseline FAYAM.  
**Données** : `memoire/06-datasets/raw/cluster_{0,4,6,8}.csv`  
**Cible** : colonne `nbrconc` (invocations concurrentes, 1 min/pas, 14 jours = 20 160 pas)

Structure de l'analyse :
1. Setup & chargement
2. Vue d'ensemble
3. Statistiques descriptives — par fonction et par cluster
4. Séries temporelles
5. Analyse des zéros
6. Distributions
7. Périodicité (ACF + FFT)
8. Stationnarité (ADF)
9. Cohérence intra-cluster
10. Comparaison inter-cluster
11. Synthèse & recommandations prétraitement

## 1 — Setup & chargement

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats, signal
from scipy.fft import rfft, rfftfreq
from statsmodels.tsa.stattools import acf, adfuller
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.dpi'] = 110

DATA_DIR = Path('../../memoire/06-datasets/raw')
CLUSTER_IDS = [0, 4, 6, 8]
CLUSTER_COLORS = {0: '#d62728', 4: '#1f77b4', 6: '#2ca02c', 8: '#ff7f0e'}

# Load all clusters: {cluster_id: DataFrame (functions × 20160)}
raw = {}
for cid in CLUSTER_IDS:
    df = pd.read_csv(DATA_DIR / f'cluster_{cid}.csv', index_col='Function')
    df.columns = df.columns.astype(int)          # timestep integers 1..20160
    raw[cid] = df
    print(f'cluster_{cid}: {df.shape[0]} fonctions {df.index.tolist()}')

# Long format per cluster: index=timestep (0..20159), columns=function IDs
long = {cid: raw[cid].T.reset_index(drop=True) for cid in CLUSTER_IDS}

# DatetimeIndex starting 2021-01-01 (synthetic, same convention as FAYAM)
time_index = pd.date_range('2021-01-01', periods=20160, freq='1min')
for cid in CLUSTER_IDS:
    long[cid].index = time_index

print('\nChargement terminé — 20 160 pas × 14 jours × 4 clusters (19 fonctions total)')

In [ ]:
# Détection environnement + chemins résultats
try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
    print('Colab détecté — montage Google Drive...')
    drive.mount('/content/drive')
    GDRIVE_RESULTS = Path('/content/drive/MyDrive/recherche-m2/experiments/eda')
    GDRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    print(f'Drive monté => {GDRIVE_RESULTS}')
except ImportError:
    IN_COLAB = False
    colab_files = None
    GDRIVE_RESULTS = None
    print('Environnement local.')

LOCAL_RESULTS = Path('../../code/experiments/eda')
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

RUN_DATE = pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M')
results = {}   # accumulateur de metriques
print(f'Run ID : {RUN_DATE}')

## 2 — Vue d'ensemble

In [ ]:
rows = []
for cid in CLUSTER_IDS:
    df = long[cid]
    for fn in df.columns:
        s = df[fn]
        zero_pct = (s == 0).mean() * 100
        b_idx = (s.std() - s.mean()) / (s.std() + s.mean()) if (s.std() + s.mean()) != 0 else np.nan
        rows.append({
            'Cluster': cid,
            'Fonction': fn,
            'Moyenne': round(s.mean(), 2),
            'Médiane': round(s.median(), 2),
            'Std': round(s.std(), 2),
            'Min': int(s.min()),
            'Max': int(s.max()),
            'CV (%)': round(s.std() / s.mean() * 100 if s.mean() != 0 else np.nan, 1),
            'Zéros (%)': round(zero_pct, 1),
            'Burstiness B': round(b_idx, 3),
        })

overview = pd.DataFrame(rows)
pd.set_option('display.max_rows', 30)
display(overview.to_string(index=False))

In [ ]:
# Summary per cluster
print('=== Résumé par cluster ===')
for cid in CLUSTER_IDS:
    sub = overview[overview['Cluster'] == cid]
    print(f"\nCluster {cid} ({len(sub)} fonctions)")
    print(f"  Moyenne globale  : {sub['Moyenne'].mean():.1f}")
    print(f"  CV moyen         : {sub['CV (%)'].mean():.1f} %")
    print(f"  Zéros moyen      : {sub['Zéros (%)'].mean():.1f} %")
    print(f"  Burstiness moyen : {sub['Burstiness B'].mean():.3f}")

In [ ]:
# Capture metriques overview
results['overview'] = overview[[
    'Cluster','Fonction','Moyenne','Std','CV (%)','Zéros (%)','Burstiness B'
]].to_dict('records')
results['cluster_means'] = {
    int(cid): round(overview[overview['Cluster']==cid]['Moyenne'].mean(), 2)
    for cid in CLUSTER_IDS
}

## 3 — Statistiques descriptives

In [ ]:
for cid in CLUSTER_IDS:
    df = long[cid]
    print(f'\n{'='*55}')
    print(f'Cluster {cid} — describe() par fonction')
    print('='*55)
    desc = df.describe().T
    desc.index.name = 'Fonction'
    desc = desc.round(2)
    display(desc)

## 4 — Séries temporelles

In [ ]:
# 4a — Vue complète 14 jours, une ligne par fonction
for cid in CLUSTER_IDS:
    df = long[cid]
    n_fn = df.shape[1]
    color = CLUSTER_COLORS[cid]
    fig, axes = plt.subplots(n_fn, 1, figsize=(16, 2.5 * n_fn), sharex=True)
    if n_fn == 1:
        axes = [axes]
    fig.suptitle(f'Cluster {cid} — série complète (14 jours)', fontsize=13, y=1.01)
    for ax, fn in zip(axes, df.columns):
        ax.plot(df.index, df[fn], lw=0.5, color=color, alpha=0.85)
        ax.set_ylabel(f'Fn {fn}', fontsize=9)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# 4b — Zoom 3 jours (jours 1-3)
zoom_end = '2021-01-04'
for cid in CLUSTER_IDS:
    df = long[cid].loc[:'2021-01-04']
    n_fn = df.shape[1]
    color = CLUSTER_COLORS[cid]
    fig, axes = plt.subplots(n_fn, 1, figsize=(16, 2.5 * n_fn), sharex=True)
    if n_fn == 1:
        axes = [axes]
    fig.suptitle(f'Cluster {cid} — zoom 3 jours', fontsize=13, y=1.01)
    for ax, fn in zip(axes, df.columns):
        ax.plot(df.index, df[fn], lw=1.0, color=color)
        ax.set_ylabel(f'Fn {fn}', fontsize=9)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    axes[-1].set_xlabel('Heure')
    plt.tight_layout()
    plt.show()

In [ ]:
# 4c — Heatmap 14 jours × 1440 minutes/jour, par fonction
for cid in CLUSTER_IDS:
    df = long[cid]
    n_fn = df.shape[1]
    fig, axes = plt.subplots(1, n_fn, figsize=(5 * n_fn, 5))
    if n_fn == 1:
        axes = [axes]
    fig.suptitle(f'Cluster {cid} — heatmap (14 jours × 1 440 min)', fontsize=12)
    day_labels = [f'J{i+1}' for i in range(14)]
    for ax, fn in zip(axes, df.columns):
        mat = df[fn].values.reshape(14, 1440)     # 14 rows = days, 1440 cols = minutes
        im = ax.imshow(mat, aspect='auto', origin='upper',
                       cmap='YlOrRd', interpolation='nearest')
        ax.set_title(f'Fn {fn}', fontsize=10)
        ax.set_xlabel('Minute du jour (0-1439)')
        ax.set_ylabel('Jour')
        ax.set_yticks(range(14))
        ax.set_yticklabels(day_labels, fontsize=8)
        plt.colorbar(im, ax=ax, shrink=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
# 4d — Profil journalier moyen (moyenne intra-cluster par minute du jour)
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()
minutes = np.arange(1440)
for ax, cid in zip(axes, CLUSTER_IDS):
    df = long[cid]
    color = CLUSTER_COLORS[cid]
    for fn in df.columns:
        daily_avg = df[fn].values.reshape(14, 1440).mean(axis=0)
        ax.plot(minutes, daily_avg, lw=1.0, alpha=0.6, label=f'Fn {fn}')
    cluster_avg = np.column_stack([
        df[fn].values.reshape(14, 1440).mean(axis=0) for fn in df.columns
    ]).mean(axis=1)
    ax.plot(minutes, cluster_avg, lw=2.5, color=color, label='Moy. cluster', zorder=5)
    ax.set_title(f'Cluster {cid} — profil journalier moyen', fontsize=11)
    ax.set_xlabel('Minute du jour')
    ax.set_ylabel('Invocations concurrentes')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(
        lambda x, _: f'{int(x)//60:02d}h{int(x)%60:02d}'
    ))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 5 — Analyse des zéros

In [ ]:
def max_consecutive_zeros(s):
    """Return max run length of consecutive zeros in series s."""
    max_run = 0
    current = 0
    for v in s:
        if v == 0:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run

zero_rows = []
for cid in CLUSTER_IDS:
    df = long[cid]
    for fn in df.columns:
        s = df[fn]
        n_zero = (s == 0).sum()
        pct_zero = n_zero / len(s) * 100
        max_cons = max_consecutive_zeros(s.values)
        zero_rows.append({
            'Cluster': cid,
            'Fonction': fn,
            'Nb zéros': int(n_zero),
            'Zéros (%)': round(pct_zero, 2),
            'Zéros consécutifs max (min)': int(max_cons),
            'Zéros consécutifs max (h)': round(max_cons / 60, 1),
        })

zero_df = pd.DataFrame(zero_rows)
display(zero_df.to_string(index=False))

In [ ]:
# Capture analyse zeros
results['zeros'] = zero_df[[
    'Cluster','Fonction','Zéros (%)','Zéros consécutifs max (h)'
]].to_dict('records')

In [ ]:
# Visualisation : taux de zéros par fonction, coloré par cluster
fig, ax = plt.subplots(figsize=(14, 5))
x_labels = [f"C{row['Cluster']}\nFn{row['Fonction']}" for _, row in zero_df.iterrows()]
colors_bar = [CLUSTER_COLORS[row['Cluster']] for _, row in zero_df.iterrows()]
bars = ax.bar(x_labels, zero_df['Zéros (%)'], color=colors_bar, edgecolor='white', alpha=0.85)
ax.set_title('Taux de zéros par fonction', fontsize=12)
ax.set_ylabel('Zéros (%)')
ax.set_xlabel('Cluster / Fonction')
for bar, val in zip(bars, zero_df['Zéros (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
# Legend
handles = [plt.Rectangle((0,0),1,1, color=CLUSTER_COLORS[c]) for c in CLUSTER_IDS]
ax.legend(handles, [f'Cluster {c}' for c in CLUSTER_IDS], loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des longueurs de runs de zéros (clusters 6 et 8 uniquement)
def zero_run_lengths(s):
    runs = []
    current = 0
    for v in s:
        if v == 0:
            current += 1
        else:
            if current > 0:
                runs.append(current)
            current = 0
    if current > 0:
        runs.append(current)
    return runs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cid in zip(axes, [6, 8]):
    df = long[cid]
    for fn in df.columns:
        runs = zero_run_lengths(df[fn].values)
        if runs:
            ax.hist(runs, bins=30, alpha=0.5, label=f'Fn {fn}', edgecolor='white')
    ax.set_title(f'Cluster {cid} — longueur des runs de zéros')
    ax.set_xlabel('Durée (minutes)')
    ax.set_ylabel('Fréquence')
    ax.legend(fontsize=8)
plt.suptitle('Distribution des plages silencieuses (zéros consécutifs)', fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Distributions

In [ ]:
# Histogrammes par fonction + densité KDE
for cid in CLUSTER_IDS:
    df = long[cid]
    n_fn = df.shape[1]
    color = CLUSTER_COLORS[cid]
    fig, axes = plt.subplots(1, n_fn, figsize=(5 * n_fn, 4))
    if n_fn == 1:
        axes = [axes]
    fig.suptitle(f'Cluster {cid} — distributions', fontsize=12)
    for ax, fn in zip(axes, df.columns):
        s = df[fn].values
        s_nonzero = s[s > 0]
        ax.hist(s_nonzero, bins=60, density=True, color=color, alpha=0.65, edgecolor='white')
        if len(s_nonzero) > 10:
            kde = stats.gaussian_kde(s_nonzero)
            xs = np.linspace(s_nonzero.min(), s_nonzero.max(), 300)
            ax.plot(xs, kde(xs), lw=2, color='black', label='KDE')
        ax.set_title(f'Fn {fn}\n(n={len(s_nonzero):,} non-nuls)', fontsize=9)
        ax.set_xlabel('Invocations concurrentes')
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plots par cluster (valeurs non-nulles, échelle log si nécessaire)
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
for ax, cid in zip(axes, CLUSTER_IDS):
    df = long[cid]
    data = [df[fn].values[df[fn].values > 0] for fn in df.columns]
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops={'color': 'black', 'lw': 2})
    color = CLUSTER_COLORS[cid]
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_title(f'Cluster {cid}', fontsize=11)
    ax.set_xticklabels([f'Fn {fn}' for fn in df.columns], fontsize=8)
    ax.set_ylabel('Invocations (non-nulles)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    # Use log scale if range is very large
    all_vals = np.concatenate(data)
    if all_vals.max() / (all_vals.min() + 1) > 100:
        ax.set_yscale('log')
        ax.set_ylabel('Invocations (log, non-nulles)')
plt.suptitle('Box plots des distributions par fonction', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Skewness et kurtosis par fonction
sk_rows = []
for cid in CLUSTER_IDS:
    df = long[cid]
    for fn in df.columns:
        s = df[fn].values
        sk_rows.append({
            'Cluster': cid, 'Fonction': fn,
            'Skewness': round(stats.skew(s), 3),
            'Kurtosis (excess)': round(stats.kurtosis(s), 3),
        })
display(pd.DataFrame(sk_rows).to_string(index=False))

## 7 — Périodicité (ACF + FFT)

In [ ]:
# ACF jusqu'à 2 jours (2880 lags) par fonction
ACF_LAGS = 2880   # 2 jours
for cid in CLUSTER_IDS:
    df = long[cid]
    n_fn = df.shape[1]
    color = CLUSTER_COLORS[cid]
    fig, axes = plt.subplots(n_fn, 1, figsize=(16, 3 * n_fn), sharex=True)
    if n_fn == 1:
        axes = [axes]
    fig.suptitle(f'Cluster {cid} — ACF (0–{ACF_LAGS} lags = 0–48h)', fontsize=12)
    lags_arr = np.arange(ACF_LAGS + 1)
    for ax, fn in zip(axes, df.columns):
        s = df[fn].values.astype(float)
        acf_vals = acf(s, nlags=ACF_LAGS, fft=True)
        ax.plot(lags_arr, acf_vals, lw=0.8, color=color)
        ax.axhline(0, color='black', lw=0.5)
        # Mark 24h and 48h
        for mark, label in [(1440, '24h'), (2880, '48h')]:
            if mark <= ACF_LAGS:
                ax.axvline(mark, color='grey', lw=1, ls='--', alpha=0.7)
                ax.text(mark + 20, ax.get_ylim()[1] * 0.85, label, fontsize=8, color='grey')
        ax.set_ylabel(f'Fn {fn}', fontsize=9)
        ax.set_ylim(-0.3, 1.05)
    axes[-1].set_xlabel('Lag (minutes)')
    plt.tight_layout()
    plt.show()

In [ ]:
# FFT Periodogram par fonction — fréquences dominantes
def top_periods(s, n_top=5):
    """Return top-n dominant periods (in minutes) from FFT."""
    s = s - s.mean()                          # remove DC
    freqs = rfftfreq(len(s), d=1.0)           # cycles per minute
    power = np.abs(rfft(s))**2
    # Exclude DC (freq=0)
    power[0] = 0
    top_idx = np.argsort(power)[::-1][:n_top]
    return [(round(1/freqs[i], 1) if freqs[i] > 0 else np.inf, round(power[i]/power.sum()*100, 2))
            for i in top_idx]

print('Périodes dominantes (FFT) par fonction\n')
for cid in CLUSTER_IDS:
    df = long[cid]
    print(f'Cluster {cid}')
    for fn in df.columns:
        periods = top_periods(df[fn].values.astype(float), n_top=5)
        pstr = ', '.join([f'{p:.0f}min ({pct:.1f}%)' for p, pct in periods])
        print(f'  Fn {fn}: {pstr}')
    print()

In [ ]:
# Capture FFT top periodes
fft_summary = {}
for cid in CLUSTER_IDS:
    df = long[cid]
    fft_summary[str(cid)] = {}
    for fn in df.columns:
        periods = top_periods(df[fn].values.astype(float), n_top=3)
        fft_summary[str(cid)][str(fn)] = [
            {'period_min': p, 'power_pct': pct} for p, pct in periods
        ]
results['fft_top_periods'] = fft_summary

In [ ]:
# Périodogramme FFT (graphique) — une figure par cluster
for cid in CLUSTER_IDS:
    df = long[cid]
    n_fn = df.shape[1]
    color = CLUSTER_COLORS[cid]
    fig, axes = plt.subplots(n_fn, 1, figsize=(16, 3 * n_fn), sharex=True)
    if n_fn == 1:
        axes = [axes]
    fig.suptitle(f'Cluster {cid} — Périodogramme FFT (périodes > 10 min)', fontsize=12)
    for ax, fn in zip(axes, df.columns):
        s = df[fn].values.astype(float) - df[fn].mean()
        freqs = rfftfreq(len(s), d=1.0)
        power = np.abs(rfft(s))**2
        # Convert to period in minutes, filter out very short periods
        mask = (freqs > 0) & (freqs < 0.1)  # period > 10 min
        periods = 1.0 / freqs[mask]
        ax.semilogy(periods, power[mask], lw=0.8, color=color)
        for mark, label in [(1440, '24h'), (720, '12h'), (480, '8h'), (60, '1h')]:
            ax.axvline(mark, color='grey', lw=1, ls='--', alpha=0.7)
            ax.text(mark * 1.02, ax.get_ylim()[1] * 0.7 if len(ax.get_ylim()) == 2 else 1,
                    label, fontsize=8, color='grey')
        ax.set_ylabel(f'Fn {fn}', fontsize=9)
        ax.set_xlim(10, 20160)
    axes[-1].set_xlabel('Période (minutes)')
    plt.tight_layout()
    plt.show()

## 8 — Stationnarité (test ADF)

In [ ]:
adf_rows = []
for cid in CLUSTER_IDS:
    df = long[cid]
    for fn in df.columns:
        s = df[fn].values.astype(float)
        try:
            result = adfuller(s, autolag='AIC')
            adf_stat, p_val, n_lags, n_obs = result[0], result[1], result[2], result[3]
            stationary = 'Oui' if p_val < 0.05 else 'Non'
        except Exception as e:
            adf_stat, p_val, n_lags = np.nan, np.nan, np.nan
            stationary = 'Erreur'
        adf_rows.append({
            'Cluster': cid, 'Fonction': fn,
            'ADF stat': round(adf_stat, 4) if not np.isnan(adf_stat) else 'N/A',
            'p-value': round(p_val, 4) if not np.isnan(p_val) else 'N/A',
            'Lags AIC': n_lags,
            'Stationnaire (p<0.05)': stationary,
        })

adf_df = pd.DataFrame(adf_rows)
display(adf_df.to_string(index=False))
print(f"\n{(adf_df['Stationnaire (p<0.05)'] == 'Oui').sum()} / {len(adf_df)} fonctions stationnaires")

In [ ]:
# Capture stationnarite ADF
results['adf'] = adf_df.to_dict('records')
results['n_stationary'] = int((adf_df['Stationnaire (p<0.05)'] == 'Oui').sum())
results['n_total_functions'] = len(adf_df)

In [ ]:
# Pour les séries non-stationnaires : test ADF après différenciation d'ordre 1
non_stat = adf_df[adf_df['Stationnaire (p<0.05)'] == 'Non']
if len(non_stat) > 0:
    print('Test ADF après différenciation d\'ordre 1 :\n')
    for _, row in non_stat.iterrows():
        cid, fn = int(row['Cluster']), row['Fonction']
        s = long[cid][fn].values.astype(float)
        s_diff = np.diff(s)
        result = adfuller(s_diff, autolag='AIC')
        p_val = result[1]
        print(f'  C{cid}/Fn{fn} → diff(1): ADF={result[0]:.4f}, p={p_val:.4f} — '
              f'Stationnaire: {"Oui" if p_val < 0.05 else "Non"}')
else:
    print('Toutes les séries sont déjà stationnaires.')

## 9 — Cohérence intra-cluster

In [ ]:
# Matrices de corrélation Pearson et Spearman par cluster
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for col_idx, cid in enumerate(CLUSTER_IDS):
    df = long[cid].copy()
    df.columns = [f'Fn{fn}' for fn in df.columns]
    # Pearson
    corr_p = df.corr(method='pearson')
    ax = axes[0, col_idx]
    mask = np.triu(np.ones_like(corr_p, dtype=bool), k=1)
    sns.heatmap(corr_p, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=-1, vmax=1, square=True, linewidths=0.5,
                cbar=col_idx == 3)
    ax.set_title(f'C{cid} — Pearson', fontsize=10)
    # Spearman
    corr_s = df.corr(method='spearman')
    ax2 = axes[1, col_idx]
    sns.heatmap(corr_s, ax=ax2, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=-1, vmax=1, square=True, linewidths=0.5,
                cbar=col_idx == 3)
    ax2.set_title(f'C{cid} — Spearman', fontsize=10)
axes[0, 0].set_ylabel('Pearson', fontsize=11)
axes[1, 0].set_ylabel('Spearman', fontsize=11)
plt.suptitle('Cohérence intra-cluster — corrélations entre fonctions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Distance DTW légère (sur sous-échantillon) pour confirmer la cohérence
# Note : sklearn n'a pas DTW natif ; on utilise une distance euclidienne normalisée comme proxy
from sklearn.preprocessing import MinMaxScaler

print('Distance euclidienne normalisée inter-fonctions (proxy cohérence cluster)\n')
for cid in CLUSTER_IDS:
    df = long[cid]
    scaler = MinMaxScaler()
    normed = pd.DataFrame(
        scaler.fit_transform(df),
        index=df.index, columns=df.columns
    )
    fn_list = df.columns.tolist()
    print(f'Cluster {cid}:')
    for i, fn_a in enumerate(fn_list):
        for fn_b in fn_list[i+1:]:
            dist = np.sqrt(((normed[fn_a] - normed[fn_b])**2).mean())
            print(f'  Fn{fn_a} ↔ Fn{fn_b} : {dist:.4f}')
    print()

## 10 — Comparaison inter-cluster

In [ ]:
# 10a — CV et burstiness par cluster (violin)
inter_rows = []
for cid in CLUSTER_IDS:
    df = long[cid]
    for fn in df.columns:
        s = df[fn]
        mean_v = s.mean()
        std_v = s.std()
        cv = std_v / mean_v * 100 if mean_v != 0 else np.nan
        burst = (std_v - mean_v) / (std_v + mean_v) if (std_v + mean_v) != 0 else np.nan
        inter_rows.append({'Cluster': f'C{cid}', 'CV (%)': cv, 'Burstiness': burst})

inter_df = pd.DataFrame(inter_rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

palette = {f'C{c}': CLUSTER_COLORS[c] for c in CLUSTER_IDS}
for ax, metric in zip(axes, ['CV (%)', 'Burstiness']):
    sns.boxplot(data=inter_df, x='Cluster', y=metric, ax=ax, palette=palette,
                order=[f'C{c}' for c in CLUSTER_IDS])
    sns.stripplot(data=inter_df, x='Cluster', y=metric, ax=ax,
                  color='black', size=7, order=[f'C{c}' for c in CLUSTER_IDS])
    ax.set_title(f'{metric} par cluster', fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Variabilité et burstiness inter-cluster', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 10b — Profils journaliers moyens normalisés (overlay inter-cluster)
fig, ax = plt.subplots(figsize=(16, 6))
minutes = np.arange(1440)
for cid in CLUSTER_IDS:
    df = long[cid]
    color = CLUSTER_COLORS[cid]
    # Aggregate: mean across all functions in the cluster, then daily mean
    all_daily = []
    for fn in df.columns:
        daily = df[fn].values.reshape(14, 1440).mean(axis=0)
        daily_norm = (daily - daily.min()) / (daily.max() - daily.min() + 1e-9)
        all_daily.append(daily_norm)
    cluster_mean_norm = np.mean(all_daily, axis=0)
    ax.plot(minutes, cluster_mean_norm, lw=2.5, color=color,
            label=f'Cluster {cid} ({len(df.columns)} fn)')

ax.set_title('Profils journaliers moyens normalisés — comparaison inter-cluster', fontsize=12)
ax.set_xlabel('Minute du jour')
ax.set_ylabel('Invocations normalisées [0-1]')
ax.xaxis.set_major_formatter(plt.FuncFormatter(
    lambda x, _: f'{int(x)//60:02d}h{int(x)%60:02d}'
))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 10c — Statistiques inter-cluster résumées
print(f'{'Cluster':>10} | {'Fonctions':>10} | {'Moy. globale':>14} | {'CV moy. (%)':>12} | '
      f'{"Zéros moy. (%)":>16} | {"Burstiness moy.":>16}')
print('-' * 90)
for cid in CLUSTER_IDS:
    sub = overview[overview['Cluster'] == cid]
    print(f'{cid:>10} | {len(sub):>10} | {sub["Moyenne"].mean():>14,.1f} | '
          f'{sub["CV (%)"].mean():>12.1f} | {sub["Zéros (%)"].mean():>16.1f} | '
          f'{sub["Burstiness B"].mean():>16.3f}')

## 11 — Synthèse & recommandations prétraitement

Cette section résume les observations clés et leurs implications pour le pipeline FAYAM.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         SYNTHÈSE EDA — Clusters HDBSCAN FaaS Azure Trace 2019          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  CLUSTER 0 (Fn 942-944) — Charges élevées                               ║
║  • Magnitude : ~100 000-400 000 invocations/min → ordres de grandeur    ║
║    très supérieurs aux autres clusters                                   ║
║  • Aucun zéro → pas de zero-inflation à gérer                           ║
║  • Périodicité 24h forte (ACF/FFT) → pattern journalier clair           ║
║  • Candidat idéal pour la reproduction baseline (signal propre)         ║
║                                                                          ║
║  CLUSTER 4 (Fn 949-954) — Charges moyennes                              ║
║  • Magnitude : ~50-200 invocations/min                                  ║
║  • Zéros minoritaires (< 5%) → traitables sans imputation lourde        ║
║  • Cluster de référence intermédiaire (ni trop simple, ni trop sparse)  ║
║                                                                          ║
║  CLUSTER 6 (Fn 138-144) — Charges faibles / zero-inflatées              ║
║  • Magnitude : ~0-20 invocations/min → séries sparse                   ║
║  • Taux de zéros élevé → imputation ou traitement spécifique            ║
║  • Fonctions peu fréquentes : cold start intense probable               ║
║  • Défi XAI : les plages de silence informent autant que les pics       ║
║                                                                          ║
║  CLUSTER 8 (Fn 964-977) — Charges faibles / bursty                      ║
║  • Magnitude : ~0-50 invocations/min avec pics sporadiques              ║
║  • Burstiness élevé → variance très forte relativement à la moyenne     ║
║  • Risque sous-représentation des pics dans le train set                ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  RECOMMANDATIONS PRÉTRAITEMENT (pipeline tsf_transf.py)                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║  1. Normalisation : MinMaxScaler ou StandardScaler par fonction          ║
║     (clusters 0 et 4 incomparables en absolu avec 6 et 8)               ║
║  2. Zéros (clusters 6 et 8) : conserver les zéros natifs —              ║
║     le TimeSeriesTransformer tolère les zéros                           ║
║  3. context_length=240 min couvre ~2 cycles de pointe journaliers       ║
║     → vérifier que la fenêtre capture bien les patterns à 24h            ║
║  4. prediction_length=120 min = 2h → horizon adapté aux pics FaaS       ║
║  5. output_attentions=True à activer dès le 1er run (matériel H1/H3)   ║
║  6. Recommandation ordre d'entraînement :                                ║
║     Cluster 0 (signal propre) → C4 → C8 → C6 (complexité croissante)   ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Tableau de bord final : récap visuel par fonction
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
metrics_to_plot = ['Moyenne', 'CV (%)', 'Zéros (%)', 'Burstiness B']
metric_labels = ['Moyenne des invocations', 'Coefficient de variation (%)',
                 'Taux de zéros (%)', 'Indice de burstiness B']

fn_labels = [f"C{row['Cluster']}\nFn{row['Fonction']}" for _, row in overview.iterrows()]
fn_colors = [CLUSTER_COLORS[row['Cluster']] for _, row in overview.iterrows()]

for row_idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    vals = overview[metric].values.astype(float)
    ax_main = axes[row_idx, 0]
    bars = ax_main.bar(range(len(overview)), vals, color=fn_colors, edgecolor='white', alpha=0.85)
    ax_main.set_xticks(range(len(overview)))
    ax_main.set_xticklabels(fn_labels, fontsize=7)
    ax_main.set_title(label, fontsize=10)
    ax_main.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    # Per cluster breakdown
    for col_idx, cid in enumerate(CLUSTER_IDS):
        sub = overview[overview['Cluster'] == cid]
        ax = axes[row_idx, col_idx + 0]  # reuse same axes
    # Legend on first column
    if row_idx == 0:
        handles = [plt.Rectangle((0,0),1,1, color=CLUSTER_COLORS[c]) for c in CLUSTER_IDS]
        ax_main.legend(handles, [f'Cluster {c}' for c in CLUSTER_IDS],
                       loc='upper right', fontsize=8)

# Remove unused axes
for r in range(4):
    for c in range(1, 4):
        axes[r, c].set_visible(False)

plt.suptitle('Tableau de bord EDA — toutes fonctions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter CV vs Burstiness — position de chaque fonction
fig, ax = plt.subplots(figsize=(10, 7))
for _, row in overview.iterrows():
    c = CLUSTER_COLORS[row['Cluster']]
    ax.scatter(row['CV (%)'], row['Burstiness B'], color=c, s=120, zorder=3, alpha=0.85)
    ax.annotate(f"C{row['Cluster']}/Fn{row['Fonction']}",
                (row['CV (%)'], row['Burstiness B']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Coefficient de variation CV (%)', fontsize=11)
ax.set_ylabel('Indice de burstiness B = (σ-μ)/(σ+μ)', fontsize=11)
ax.set_title('Positionnement CV × Burstiness — profil de variabilité par fonction', fontsize=12)
handles = [plt.scatter([], [], color=CLUSTER_COLORS[c], s=80, label=f'Cluster {c}') for c in CLUSTER_IDS]
ax.legend(handles=handles, fontsize=10)
ax.annotate('B > 0 : plus bursty que Poisson', xy=(ax.get_xlim()[0]*1.01, 0.02), fontsize=9, color='grey')
ax.annotate('B < 0 : plus régulier que Poisson', xy=(ax.get_xlim()[0]*1.01, -0.06), fontsize=9, color='grey')
plt.tight_layout()
plt.show()

## 12 — Sauvegarde des résultats

In [ ]:
import json as _json

results['run_date'] = RUN_DATE
results['clusters_analyzed'] = CLUSTER_IDS
results['n_functions'] = sum(long[c].shape[1] for c in CLUSTER_IDS)
results['n_timesteps'] = 20160

json_fname = f'eda_results_{RUN_DATE}.json'
_payload = _json.dumps(results, indent=2, default=str)

# 1 — sauvegarde locale (toujours)
local_json = LOCAL_RESULTS / json_fname
local_json.write_text(_payload, encoding='utf-8')
print(f'JSON local : {local_json}')

# 2 — sauvegarde Drive (Colab uniquement)
if IN_COLAB and GDRIVE_RESULTS:
    drive_json = GDRIVE_RESULTS / json_fname
    drive_json.write_text(_payload, encoding='utf-8')
    print(f'JSON Drive : {drive_json}')

# 3 — mise a jour REGISTER.md
register_path = (GDRIVE_RESULTS if IN_COLAB and GDRIVE_RESULTS else LOCAL_RESULTS) / 'REGISTER.md'
_cm = results.get('cluster_means', {})
_n_stat = results.get('n_stationary', '?')
_n_tot  = results.get('n_total_functions', '?')
_row = (
    f'| {RUN_DATE} '
    f'| {json_fname} '
    f'| {_cm.get(0,"?"):,} '
    f'| {_cm.get(4,"?"):,} '
    f'| {_cm.get(6,"?")} '
    f'| {_cm.get(8,"?")} '
    f'| {_n_stat}/{_n_tot} '
    f'| — |\n'
)
_header = (
    '# Registre EDA — Clusters FaaS\n\n'
    '> Mis a jour automatiquement par `EDA_clusters.ipynb`.\n\n'
    '| Date run | Fichier JSON | C0 moy. | C4 moy. | C6 moy. | C8 moy. | Fn stationnaires | Notes |\n'
    '|----------|-------------|---------|---------|---------|---------|-----------------|-------|\n'
)
if register_path.exists():
    _existing = register_path.read_text(encoding='utf-8')
    _existing = _existing.replace('| *(premier run a venir)* | -- | -- | -- | -- | -- | -- | -- |\n', '')
    register_path.write_text(_existing.rstrip('\n') + '\n' + _row, encoding='utf-8')
else:
    register_path.write_text(_header + _row, encoding='utf-8')
print(f'REGISTER.md mis a jour : {register_path}')
print('=== Sauvegarde terminee ===')

In [ ]:
# Telechargement navigateur (Colab uniquement)
if IN_COLAB:
    colab_files.download(str(GDRIVE_RESULTS / json_fname))
    print(f'Telechargement declenche : {json_fname}')
else:
    print(f'Local — fichier disponible dans : {local_json}')

## Etape finale — Archiver le notebook avec ses sorties

> **A faire apres 'Run All'**, avant de fermer Colab :

1. **Fichier → Telecharger → Telecharger .html** — contient toutes les figures + tableaux
2. **Fichier → Telecharger → Telecharger .ipynb** — contient les outputs dans le JSON (optionnel)
3. Glisse le `.html` dans `code/experiments/eda/` sur ta machine locale
4. Le JSON de metriques a deja ete telecharge automatiquement par la cellule precedente

Le fichier HTML est consultable dans n'importe quel navigateur, sans Python ni Jupyter.